# Notebook 1: Audio I/O and Preprocessing

**What:** How Demucs loads, resamples, and segments audio.

**Why:** Training and inference need consistent sample rate (44.1kHz), channel handling, and segment extraction.

**How:** Use `demucs.audio`, `torchaudio`, and understand segment logic.

## 1. Load Audio with torchaudio

Demucs expects: `(channels, samples)` at 44.1 kHz stereo.

In [2]:
import sys
sys.path.insert(0, r'D:\demucs')

import torch
import torchaudio
from pathlib import Path

# Create a short test file if you don't have one
test_path = Path("test_audio\修炼爱情.wav")
if not test_path.exists():
    # Generate 2 seconds of stereo noise
    sr = 44100
    audio = torch.randn(2, sr * 2) * 0.3
    torchaudio.save(str(test_path), audio, sr)
    print("Created test_audio.wav")

wav, sr = torchaudio.load(str(test_path))
print(f"Shape: {wav.shape} (channels, samples)")
print(f"Sample rate: {sr}")
print(f"Duration: {wav.shape[1] / sr:.2f} s")

Shape: torch.Size([2, 12663744]) (channels, samples)
Sample rate: 44100
Duration: 287.16 s


## 2. Resampling and Channel Conversion

Demucs uses `julius` (or `torchaudio.transforms.Resample`) for resampling. Target: 44100 Hz, stereo.

In [5]:
from demucs.audio import convert_audio_channels
import julius

# ─── WHAT: Resample audio to a target sample rate if needed ─────────────────
# WHY: Demucs expects 44.1 kHz. Different sample rates (e.g. 48 kHz, 22.05 kHz)
#      would change STFT time/freq resolution and break the pretrained model.
# HOW: Julius uses sinc interpolation (high-quality, differentiable, GPU-capable).
def resample_if_needed(wav, sr, target_sr=44100):
    if sr != target_sr:
        wav = julius.resample_frac(wav, sr, target_sr)  # (C, T) → (C, T') where T' = T * target_sr / sr
    return wav

# ─── WHAT: Convert mono → stereo (or pass through if already stereo) ──────────
# WHY: Demucs model expects 2 channels. Mono input would mismatch model input dims.
#      Duplicating mono to L/R preserves content; multi-ch → take first 2.
# HOW: convert_audio_channels uses mean for downmix, expand for upmix, or slicing.
def to_stereo(wav):
    """Convert to 2 channels if mono."""
    return convert_audio_channels(wav, 2)

# ─── Pipeline: resample first (change sample count), then fix channels ────────
# Order matters: resample operates on (C, T); channel count is independent.
wav_processed = resample_if_needed(wav, sr)
wav_processed = to_stereo(wav_processed)
print(f"After processing: {wav_processed.shape}, 44100 Hz stereo")

After processing: torch.Size([2, 12663744]), 44100 Hz stereo


## 3. Segment Extraction (Training Logic)

Training samples a random segment of length `segment * samplerate` from each track. This is how `wav.py` / dataset works conceptually.

In [6]:
def extract_segment(wav, segment_samples, random_start=True):
    """
    Extract a segment from audio.
    wav: (C, T)
    segment_samples: length in samples
    """
    C, T = wav.shape
    if T <= segment_samples:
        return wav
    if random_start:
        start = torch.randint(0, T - segment_samples + 1, (1,)).item()
    else:
        start = 0
    return wav[:, start:start + segment_samples]

segment_sec = 4
sr = 44100
segment = extract_segment(wav_processed, segment_sec * sr, random_start=False)
print(f"Segment: {segment.shape} = {segment.shape[1]/sr:.2f} seconds")

Segment: torch.Size([2, 176400]) = 4.00 seconds


## 4. Demucs AudioFile (for arbitrary formats)

For MP3, FLAC, etc., Demucs uses `demucs.audio.AudioFile` + ffmpeg. Requires ffmpeg on PATH.

In [7]:
# ─── WHAT: Try to use Demucs' AudioFile for arbitrary audio formats (MP3, FLAC, M4A, etc.) ─
# WHY: torchaudio mainly supports WAV/FLAC/MP3 on some platforms. AudioFile uses ffmpeg to
#      decode virtually any format, and can resample/remix on the fly.
# HOW: AudioFile calls ffprobe for metadata, then ffmpeg to decode to raw float32 PCM.
#      Requires ffmpeg (and ffprobe) installed and on system PATH.
try:
    from demucs.audio import AudioFile
    # Wrap path as AudioFile — no actual read yet; just validates path exists
    af = AudioFile(test_path)
    print(f"AudioFile: {af}")
    # read() decodes via ffmpeg, resamples to 44100 Hz, converts to 2 channels.
    # Returns torch tensor: (C, T) for single stream, or (S, C, T) for multiple streams.
    data = af.read(samplerate=44100, channels=2)
    print(f"Read shape: {data.shape}")
except Exception as e:
    # ffmpeg not found on PATH, or other I/O/metadata error
    print(f"AudioFile needs ffmpeg: {e}")

AudioFile: AudioFile(path=test_audio\修炼爱情.wav, samplerate=44100, channels=2, streams=1)
Read shape: torch.Size([1, 2, 12663744])


## 5. Memory Estimate for 3070 Ti

For a segment of S seconds at 44.1kHz stereo:
- Samples per channel: 44100 * S
- Shape: (2, 44100*S)
- Float32: 4 bytes per sample → ~0.35 MB per second of stereo.

Segment 8s × batch 8 ≈ 2.8 MB input. Model weights + activations dominate; use small batches.

In [8]:
segment_sec = 8
batch = 8
channels = 2
samples = 44100 * segment_sec
bytes_per_sample = 4
mb = (channels * samples * batch * bytes_per_sample) / 1e6
print(f"Batch input (8s, batch=8): ~{mb:.1f} MB")

Batch input (8s, batch=8): ~22.6 MB


**Next:** Notebook 2 — Model architectures (Demucs, HDemucs, HTDemucs).